# Go2 configuration atlas and structured contact planning
Python 3.11. Upload `configuration_planning_molab_20261112_v1.zip` alongside this notebook. The atlas/kinematic animation is not a dynamic demonstration. All learning remains disabled. Existing results are included for review; new output directories are required.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,zipfile,subprocess
assert sys.version_info[:2]==(3,11)
root=Path("go2_configuration_workspace")
assert not root.exists(), "Use a new workspace"
with zipfile.ZipFile("configuration_planning_molab_20261112_v1.zip") as z:
    z.extractall(root)
os.chdir(root)
for name,digest in json.loads(Path("configuration_planning_bundle_manifest.json").read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest()==digest,name
subprocess.run([sys.executable,"-m","pip","install","-r","configuration_planning_requirements.txt"],check=True)
os.environ.update(MUJOCO_GL="egl",WANDB_MODE="disabled",MPLCONFIGDIR="/tmp/go2_config_mpl",XDG_CACHE_HOME="/tmp/go2_config_cache")
def run(*args):
    subprocess.run([sys.executable,"-m",*args],check=True)


In [ ]:
run("pytest","-q","experiments/recoverability/test_core.py","experiments/recoverability/test_transition.py","experiments/recoverability/bidirectional/test_bidirectional.py","experiments/recoverability/contact_transition/test_contact_transition.py","experiments/recoverability/configuration_planning/test_planning.py")


In [ ]:
run("experiments.recoverability.configuration_planning.atlas","--output","output/molab_configuration_atlas","--max-nfev","80")
run("experiments.recoverability.configuration_planning.connectivity","--atlas","output/molab_configuration_atlas","--output","output/molab_configuration_connectivity")


In [ ]:
run("experiments.recoverability.configuration_planning.replay","--source","output/configuration_shooting_contact_refined_20261112_v1","--output","output/molab_configuration_replay","--seed-base","202635120")
print(Path("output/molab_configuration_replay/learning_gate.json").read_text())


In [ ]:
run("experiments.recoverability.configuration_planning.report","--atlas","output/molab_configuration_atlas","--connectivity","output/molab_configuration_connectivity","--plans","output/configuration_shooting_20261112_v1","--refined","output/configuration_shooting_refined_20261112_v1","--contact-refined","output/configuration_shooting_contact_refined_20261112_v1","--replay","output/molab_configuration_replay","--reverse","output/configuration_reverse_20261112_v1","--audit","output/configuration_audit_20261112_v1","--output","output/molab_configuration_report")
print(Path("output/molab_configuration_report/measured_report.md").read_text())


Optional fresh structured planning. This uses deterministic multiple shooting, not random action-sequence sampling. Optimizer state nodes never appear in uninterrupted validation.

In [ ]:
RUN_NEW_PLANNING=False
if RUN_NEW_PLANNING:
    run("experiments.recoverability.configuration_planning.plan","--atlas","output/molab_configuration_atlas","--output","output/molab_configuration_plans","--max-nfev","12")
    run("experiments.recoverability.configuration_planning.refine","--source","output/molab_configuration_plans","--output","output/molab_configuration_refined","--max-nfev","16")
    run("experiments.recoverability.configuration_planning.refine","--source","output/molab_configuration_refined","--exact-start","output/molab_configuration_plans/exact_start_D.pt","--path-constraints","--seed-base","202636120","--max-nfev","12","--output","output/molab_configuration_contact_refined")
    run("experiments.recoverability.configuration_planning.reverse","--atlas","output/molab_configuration_atlas","--output","output/molab_configuration_reverse")
    run("experiments.recoverability.configuration_planning.audit","--plans","output/molab_configuration_plans","--refined","output/molab_configuration_refined","--contact-refined","output/molab_configuration_contact_refined","--reverse","output/molab_configuration_reverse","--output","output/molab_configuration_audit")


In [ ]:
from IPython.display import Image,display
display(Image(filename="output/configuration_visualizations_20261112_v1/geometry_only_path.gif"))
display(Image(filename="output/configuration_visualizations_20261112_v1/final_frozen_D.gif"))


In [ ]:
RUN_NEW_EGL_RENDER=False
if RUN_NEW_EGL_RENDER:
    run("experiments.recoverability.configuration_planning.visualize","--plans","output/configuration_shooting_20261112_v1","--refined","output/configuration_shooting_contact_refined_20261112_v1","--connectivity","output/molab_configuration_connectivity","--output","output/molab_configuration_visualizations")


No BC/PPO entry point is enabled. The original study found a geometric/static path but no valid robust dynamic bridge. Original-metric reverse recovery successes had excessive cumulative slip and are not low-slip handoff certificates.

In [ ]:
EXPORT_NEW_RESULTS=False
if EXPORT_NEW_RESULTS:
    args=[sys.executable,"scripts/package_configuration_planning.py","--output","output/configuration_planning_export.zip"]
    for directory in ("output/molab_configuration_atlas","output/molab_configuration_connectivity","output/molab_configuration_replay","output/molab_configuration_report","output/configuration_shooting_20261112_v1","output/configuration_shooting_refined_20261112_v1","output/configuration_shooting_contact_refined_20261112_v1","output/configuration_reverse_20261112_v1","output/configuration_audit_20261112_v1","output/configuration_visualizations_20261112_v1"):
        args.extend(["--results",directory])
    subprocess.run(args,check=True)
